# Image Augmentation and Splitting

In [ ]:
# ===================================================
# 0. Setup Environment
# ===================================================

# Install required packagesfrom google.colab import drive
import os
import zipfile
import matplotlib.pyplot as plt
import numpy as np
import cv2
import random
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from PIL import Image
import shutil
from sklearn.model_selection import train_test_split

In [ ]:
!pip install split-folders
import splitfolders

In [ ]:
# Mount Google Drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# Define the path to the dataset ZIP file in Drive
zip_path = '/content/drive/MyDrive/Betel_Leaf_Raw.zip'  # Update with your actual path

In [ ]:
# Unzip the dataset
extract_path = '/content/Betel_Leaf_Disease_Raw'
with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_path)

In [ ]:
# The actual dataset path inside the extracted folder
dataset_path = os.path.join(extract_path, 'Betel_Leaf_Raw')

In [ ]:
# Count the number of images in each class subfolder
classes = sorted(os.listdir(dataset_path))  # Get class names
print("Number of images in each class:")
for class_name in classes:
    class_path = os.path.join(dataset_path, class_name)
    if os.path.isdir(class_path):
        num_images = len([f for f in os.listdir(class_path) if f.endswith(('.jpg', '.png', '.jpeg'))])
        print(f"{class_name}: {num_images} images")

In [ ]:
# Display a few images from each class
def show_images_from_class(class_name, num_images=3):
    class_path = os.path.join(dataset_path, class_name)
    image_files = [f for f in os.listdir(class_path) if f.endswith(('.jpg', '.png', '.jpeg'))]

    if len(image_files) == 0:
        print(f"No images found in {class_name}")
        return

    selected_images = random.sample(image_files, min(num_images, len(image_files)))

    fig, axes = plt.subplots(1, len(selected_images), figsize=(12, 4))
    fig.suptitle(class_name)

    for i, img_file in enumerate(selected_images):
        img_path = os.path.join(class_path, img_file)
        img = cv2.imread(img_path)
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)  # Convert BGR to RGB
        axes[i].imshow(img)
        axes[i].axis('off')

    plt.show()

In [ ]:
# Show images from each class
for class_name in classes:
    show_images_from_class(class_name)

In [ ]:
# Define augmentation parameters (best practices: rotation, shifts, shear, zoom, and horizontal flip)
datagen = ImageDataGenerator(
    rotation_range=20,          # rotate images up to 20 degrees
    width_shift_range=0.2,      # shift images horizontally by up to 20%
    height_shift_range=0.2,     # shift images vertically by up to 20%
    brightness_range=[0.8, 1.2],# vary brightness between 80% and 120%
    zoom_range=0.15,            # zoom in/out by up to 15%
    horizontal_flip=True,       # flip images horizontally
    vertical_flip=True,         # flip images vertically
    fill_mode='nearest'         # fill in new pixels
)

In [ ]:
# Define paths
original_dir = dataset_path
augmented_dir = os.path.join(extract_path, 'Betel_Leaf_Augmented')
if not os.path.exists(augmented_dir):
    os.makedirs(augmented_dir)

In [ ]:
# Create class directories in augmented folder
classes = os.listdir(original_dir)
for class_name in classes:
    class_path = os.path.join(original_dir, class_name)
    if os.path.isdir(class_path):
        augmented_class_path = os.path.join(augmented_dir, class_name)
        os.makedirs(augmented_class_path, exist_ok=True)

        # Process each image
        for image_file in os.listdir(class_path):
            image_path = os.path.join(class_path, image_file)

            # Skip non-image files
            ext = os.path.splitext(image_file)[1].lower()
            if ext not in ['.jpg', '.jpeg', '.png', '.bmp', '.gif']:
                continue

            if os.path.isfile(image_path):
                # Load image and convert to RGB
                img = Image.open(image_path).convert('RGB')
                img_array = np.array(img)

                # Generate 3 augmented images
                for i in range(1, 4):
                    augmented_img = datagen.random_transform(img_array)
                    augmented_img_pil = Image.fromarray(augmented_img.astype('uint8'))

                    # Create new filename
                    base, ext = os.path.splitext(image_file)
                    new_filename = f"{base}_aug_{i}{ext}"
                    new_filepath = os.path.join(augmented_class_path, new_filename)

                    # Save augmented image
                    augmented_img_pil.save(new_filepath)

print("Data augmentation completed successfully!")

In [ ]:
# 1. Zip the augmented directory
shutil.make_archive(base_name= '/content/Betel_Leaf_Disease_Raw/Betel_Leaf_Augmented',
                    format= 'zip',
                    root_dir= '/content/Betel_Leaf_Disease_Raw/Betel_Leaf_Augmented')

# 2. Define destination path in Google Drive
drive_destination = '/content/drive/MyDrive/Betel_Leaf_Augmented.zip'  # Adjust path if needed

# Create parent directories if needed
os.makedirs(os.path.dirname(drive_destination), exist_ok=True)

# 3. Copy the zipped file to Google Drive
# Use shutil.copy2 to preserve metadata (optional)
shutil.copy2('/content/Betel_Leaf_Disease_Raw/Betel_Leaf_Augmented.zip', drive_destination)
# Changed from shutil.copy to shutil.copy2 and updated the source path to the zip file

# Verify completion
print(f"\n✅ Augmented data zip saved to Google Drive at:\n{drive_destination}")
print(f"File size: {os.path.getsize(drive_destination)/1024/1024:.2f} MB")

In [ ]:
# ===================================================
# 1. Configuration
# ===================================================
# Paths (Verify these match your Drive structure)
DRIVE_PATH = '/content/drive/MyDrive'
RAW_ZIP = os.path.join(DRIVE_PATH, 'Betel_Leaf_Raw.zip')
AUG_ZIP = os.path.join(DRIVE_PATH, 'Betel_Leaf_Augmented.zip')
MERGED_DIR = '/content/Betel_Leaf_Disease'
FINAL_DIR = '/content/Final_Dataset'
FINAL_ZIP = '/content/Betel_Leaf_Final_Dataset.zip'
DRIVE_DEST = os.path.join(DRIVE_PATH, 'Betel_Leaf_Final_Dataset.zip')

# Parameters
TARGET_SIZE = (256, 256)  # Width, Height
SPLIT_RATIO = (0.7, 0.15, 0.15)  # Train/Test/Valid
RANDOM_SEED = 42

# ===================================================
# 2. Resize & Merge with Actual Class Handling
# ===================================================
def resize_image(img_file, save_path):
    """Resize with aspect ratio preservation and padding"""
    try:
        img = Image.open(img_file).convert('RGB')
        img.thumbnail(TARGET_SIZE, Image.Resampling.LANCZOS)

        # Create padded image
        new_img = Image.new("RGB", TARGET_SIZE, (255, 255, 255))
        new_img.paste(img, (
            (TARGET_SIZE[0] - img.size[0]) // 2,
            (TARGET_SIZE[1] - img.size[1]) // 2
        ))

        new_img.save(save_path)
        return True
    except Exception as e:
        print(f"Error processing image: {str(e)}")
        return False

def process_zip(zip_path, output_dir):
    """Handles actual class names and directory structures"""
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        for file_info in zip_ref.infolist():
            if file_info.is_dir():
                continue

            # Handle different zip structures
            if "Betel_Leaf_Raw" in file_info.filename:
                # Raw zip structure: Betel_Leaf_Raw/Class Name/image.jpg
                parts = file_info.filename.split('/')
                if len(parts) < 3:
                    continue  # Skip root files
                class_name = parts[1]
                file_name = parts[-1]
            else:
                # Augmented zip structure: Class Name/image_aug_X.jpg
                parts = file_info.filename.split('/')
                if len(parts) < 2:
                    continue
                class_name = parts[0]
                file_name = parts[-1]

            # Create class directory with actual name
            class_dir = os.path.join(output_dir, class_name)
            os.makedirs(class_dir, exist_ok=True)

            # Process and save image
            save_path = os.path.join(class_dir, file_name)
            if not os.path.exists(save_path):
                with zip_ref.open(file_info) as f:
                    if not resize_image(f, save_path):
                        print(f"Failed to process: {file_info.filename}")

def merge_and_resize():
    """Main merge function with actual class names"""
    print("⏳ Starting merge and resize...")

    # Clean directories
    shutil.rmtree(MERGED_DIR, ignore_errors=True)
    os.makedirs(MERGED_DIR, exist_ok=True)

    # Process both zips
    for zip_path in [RAW_ZIP, AUG_ZIP]:
        print(f"🔨 Processing {os.path.basename(zip_path)}...")
        process_zip(zip_path, MERGED_DIR)

    # Verification
    print("\n✅ Merge complete. Class counts:")
    for class_name in sorted(os.listdir(MERGED_DIR)):
        class_path = os.path.join(MERGED_DIR, class_name)
        count = len(os.listdir(class_path))
        print(f"  - {class_name}: {count} images")

# ===================================================
# 3. CUSTOM SPLITTING IMPLEMENTATION
# ===================================================
def split_dataset():
    print("\n⏳ Splitting dataset with custom grouping...")

    # Clean previous splits
    shutil.rmtree(FINAL_DIR, ignore_errors=True)
    os.makedirs(os.path.join(FINAL_DIR, 'train'), exist_ok=True)
    os.makedirs(os.path.join(FINAL_DIR, 'test'), exist_ok=True)
    os.makedirs(os.path.join(FINAL_DIR, 'val'), exist_ok=True)

    # Process each class
    for class_name in os.listdir(MERGED_DIR):
        class_path = os.path.join(MERGED_DIR, class_name)
        if not os.path.isdir(class_path):
            continue

        # Group images by original base name
        groups = {}
        for filename in os.listdir(class_path):
            # Extract base name (without _aug_X)
            base = filename.split('_aug_')[0]
            base = os.path.splitext(base)[0]  # Remove extension
            if base not in groups:
                groups[base] = []
            groups[base].append(filename)

        # Convert to list and shuffle
        group_list = list(groups.items())
        train_groups, temp_groups = train_test_split(
            group_list,
            test_size=0.3,
            random_state=RANDOM_SEED
        )
        test_groups, val_groups = train_test_split(
            temp_groups,
            test_size=0.5,
            random_state=RANDOM_SEED
        )

        # Copy files to splits
        for split_name, groups in [('train', train_groups),
                                   ('test', test_groups),
                                   ('val', val_groups)]:
            split_class_dir = os.path.join(FINAL_DIR, split_name, class_name)
            os.makedirs(split_class_dir, exist_ok=True)

            for base_name, files in groups:
                for file in files:
                    src = os.path.join(class_path, file)
                    dst = os.path.join(split_class_dir, file)
                    shutil.copy2(src, dst)

        print(f"\n✅ {class_name} split:")
        print(f"  - Train: {len(train_groups)} groups")
        print(f"  - Test: {len(test_groups)} groups")
        print(f"  - Val: {len(val_groups)} groups")

    # Final verification
    print("\n📊 Final counts:")
    for split in ['train', 'test', 'val']:
        split_path = os.path.join(FINAL_DIR, split)
        total = sum([len(files) for _, _, files in os.walk(split_path)])
        print(f"  - {split.capitalize()}: {total} images")

# ===================================================
# 4. Save to Drive
# ===================================================
def save_to_drive():
    """Compress and save final dataset"""
    print("\n⏳ Saving to Google Drive...")

    # Create zip
    shutil.make_archive(FINAL_ZIP[:-4], 'zip', FINAL_DIR)

    # Copy to Drive
    shutil.copy2(FINAL_ZIP, DRIVE_DEST)

    # Verify
    if os.path.exists(DRIVE_DEST):
        size_mb = os.path.getsize(DRIVE_DEST) / 1024**2
        print(f"\n✅ Success! Final dataset saved to:")
        print(f"  - Path: {DRIVE_DEST}")
        print(f"  - Size: {size_mb:.2f} MB")
    else:
        raise FileNotFoundError("Failed to save to Drive")

# ===================================================
# 5. Main Execution
# ===================================================
if __name__ == "__main__":
    # Run pipeline
    merge_and_resize()
    split_dataset()
    save_to_drive()

    # Cleanup
    shutil.rmtree(MERGED_DIR, ignore_errors=True)
    shutil.rmtree(FINAL_DIR, ignore_errors=True)
    os.remove(FINAL_ZIP)

    print("\n🚀 Pipeline completed successfully!")